In [1]:
!brew install poppler tesseract libmagic

==> Auto-updating Homebrew...
Adjust how often this is run with `$HOMEBREW_AUTO_UPDATE_SECS` or disable with
`$HOMEBREW_NO_AUTO_UPDATE=1`. Hide these hints with `$HOMEBREW_NO_ENV_HINTS=1` (see `man brew`).
Error: Failed to download https://formulae.brew.sh/api/cask.jws.json!
Failed to download https://formulae.brew.sh/api/formula_tap_migrations.jws.json!
Failed to download https://formulae.brew.sh/api/cask_tap_migrations.jws.json!
⠋ JSON API formula_tap_migrations.jws.json           Downloading   1.9KB/-------
⠋ JSON API cask.jws.json                             Downloading  15.4MB/-------
⠋ JSON API cask_tap_migrations.jws.json              Downloading   2.4KB/-------⠋ JSON API formula_tap_migrations.jws.json           Downloading   1.9KB/-------
⠋ JSON API cask.jws.json                             Downloading  15.4MB/-------
⠋ JSON API cask_tap_migrations.jws.json              Downloading   2.4KB/-------⠙ JSON API formula_tap_migrations.jws.json           Downloading   1.9KB/-------


In [2]:
%pip install -Uq "unstructured[all-docs]"
%pip install -Uq langchain_chroma
%pip install -Uq langchain-community langchain-openai
%pip install -Uq python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -Uq langchain_groq
%pip install -Uq langchain_huggingface


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [32]:
import json 
from typing import List

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
# from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage,SystemMessage
from dotenv import load_dotenv 

load_dotenv()

True

In [68]:
def partition_document(filepath): 
    """ Extract elements from the PDF usng unstructured"""
    print(f"partition the document: {filepath}")
    elements=partition_pdf(
        filename=filepath,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_to_payload=True,
        extract_image_block_types=["Images"]
    )

    print(f"Extracted  {len(elements)} elements")
    return elements

filepath="data/QSVD_paper.pdf"
# filepath="/Users/vikram/Documents/RAG/RAG/data/QSVD (2) (1).pdf"
elements=partition_document(filepath)

partition the document: data/QSVD_paper.pdf


No languages specified, defaulting to English.
The requested type (Images) doesn't match any available type


Extracted  511 elements


In [69]:
def create_chunks_by_title(elements): 
    "create intellgnet chunknusing title-based stratergy"
    print("creting smart chunks")

    chunks=chunk_by_title(
        elements,
        max_characters=3000,
        combine_text_under_n_chars=500,
        new_after_n_chars=2400
    )
    print(f"created {len(chunks)} chunks")
    return chunks

In [71]:
chunks=create_chunks_by_title(elements)

creting smart chunks
created 54 chunks


In [72]:
chunks

In [73]:
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>",
 "<class 'unstructured.documents.elements.TableChunk'>"}

In [74]:
def create_ai_enhanced_summary(text:str,tables:List[str],images: List[str])->str: 
    """create AI-emhanced summary for mixed content"""
    try: 
        llm=ChatGroq(model="llama-3.3-70b-versatile")
        prompt_text=f"""you are creating a searchable description for document content retrieval.
        CONTENT TO ANALYZE: 
        TEXT CONTENT: {text} 
           """
        if tables: 
            prompt_text+="TABLES: \n"
            for i, table in enumerate(tables): 
                prompt_text += f"Table {i+1}: \n {table}\n\n"
                prompt_text += """
                YOUR TASK: 
                Generate a comprehensive, searchable description that covers: 
                1. Key facts, numbers and data points from text and tables
                2. Main topics and concepts discussed
                3. Questions this content could answer
                4. Visual content analysis (charts,diagram,patterns in image)
                5. Alternative search terms users might use

                Make it detailed and searchable- priortize findability over brevity.

                SEARCHABLE DESCRIPTION: 
            """
        message_content=[{"type": "text","text":prompt_text}]
        for image_base64 in images: 
            message_content.append({
                "tyep":"image_url",
                "image_url":{"url":f"data:image/jpeg;base64,{image_base64}"}
            })
        message=HumanMessage(content=message_content)
        response=llm.invoke([message])
        return response.content
    except Exception as e: 
        print("AI summary fialed: {e}")

        summary=f"{text[:300]}..."
        return summary






In [75]:
def seperate_content_types(chunk): 
    """Analyze what types of content are in a chunk"""
    content_data={
        'text':chunk.text,
        'tables':[],
        'images':[],
        'types':['text']
    }

    if hasattr(chunk,'metadata') and hasattr(chunk.metadata,'orig_elements'): 
        for element in chunk.metadata.orig_elements: 
            element_type=type(element).__name__

            if element_type =="Table": 
                content_data['types'].append('table')
                table_html=getattr(element.metadata,'text_as_html',element.text)
                content_data['tables'].append(table_html)
            elif element_type=='Images': 
                if hasattr(element,'metadata') and hasattr(element.metadata,'iamges'): 
                    content_data['types'].append('iamge')
                    content_data['images'.append(element.metadata.iamge_base64)]
    content_data['types']=list(set(content_data['types']))
    return content_data


In [76]:
def summarise_chunks(chunks): 
    """process all chunks with AI summaries"""
    print(f"processing cunks with AI summaries")

    langchain_documents=[]
    total_chunks=len(chunks)

    for i,chunk in enumerate(chunks): 
        current_chunk=i+1
        print(f"processing chunk {current_chunk}/{total_chunks}")

        content_data=seperate_content_types(chunk)
        print(f"content_data : {content_data.keys}")
        print(f"types found: {content_data['types']}")
        print(f"tables: {len(content_data['tables'])},Images: {len(content_data['images'])}")

        if content_data['tables'] or content_data['images']: 
            print(f" creating AI summary for mixed content")
            try: 
                enhanced_content=create_ai_enhanced_summary(content_data['text'],content_data['tables'],content_data['images'])
                print(f" Ai summary creted successfully")
                print("enhanced content preview: {enhanced_conten[:200]}...")
            except Exception as e: 
                print(f"AI sumamry failed {e}")
                enhanced_content=content_data['text']
        else: 
            print(f" using raw text (no table/images)")
            enhanced_content=content_data['text']

        doc=Document(
            page_content=enhanced_content,
            metadata={
                'original_content':json.dumps({
                    'raw_text':content_data['text'],
                    'tables_html':content_data['tables'],
                    'images_base64': content_data['images']
                })
            }
        )
        langchain_documents.append(doc)
    
    print(f"procesed {len(langchain_documents)} chunks")
    return langchain_documents
    


            
    

In [77]:
processed_chunks=summarise_chunks(chunks)

processing cunks with AI summaries
processing chunk 1/54
content_data : <built-in method keys of dict object at 0x17dfe0500>
types found: ['text']
tables: 0,Images: 0
 using raw text (no table/images)
processing chunk 2/54
content_data : <built-in method keys of dict object at 0x17dfe0380>
types found: ['text']
tables: 0,Images: 0
 using raw text (no table/images)
processing chunk 3/54
content_data : <built-in method keys of dict object at 0x17dfe0480>
types found: ['text']
tables: 0,Images: 0
 using raw text (no table/images)
processing chunk 4/54
content_data : <built-in method keys of dict object at 0x17dfe0500>
types found: ['text']
tables: 0,Images: 0
 using raw text (no table/images)
processing chunk 5/54
content_data : <built-in method keys of dict object at 0x17dfe0380>
types found: ['text']
tables: 0,Images: 0
 using raw text (no table/images)
processing chunk 6/54
content_data : <built-in method keys of dict object at 0x17dfe0480>
types found: ['text']
tables: 0,Images: 0
 us

In [78]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "2025\\n\\narXiv:2510.16292v1 [cs.LG] 18 Oct\\n\\nQSVD: Efficient Low-rank Approximation for Unified Query-Key-Value Weight Compression in Low-Precision Vision-Language Models\\n\\nYutong Wang!* Haiyu Wang!* Sai Qian Zhang!\\u201d\\n\\n\'Tandon School of Engineering, New York University 2Courant Institute of Mathematical Sciences, New York University {yw6594, hw3689, sai.zhang}@nyu.edu\\n\\nAbstract\\n\\nVision-Language Models (VLMs) are integral to tasks such as image captioning and visual question answering, but their high computational cost, driven by large memory footprints and processing time, limits their scalability and real-time ap- plicability. In this work, we propose leveraging Singular-Value Decomposition (SVD) over the joint query (Q), key (K), and value (V) weight matrices to reduce KV cache size and computational overhead. We in addition introduce an efficient rank allocation strategy that dynamically adjusts the SVD 

In [79]:
def create_vector_store(documents,persist_directory="db/chroma_db"): 
    print("Creating embeddings and store in chroma")

    embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    print("creting vector store")

    vectorstore=Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space":'cosine'}
    )

    print("--- finished creating vector store")
    print(f" vector store created and saved to persist directory {persist_directory}")
    return vectorstore

db=create_vector_store(processed_chunks)

Creating embeddings and store in chroma


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7503.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


creting vector store
--- finished creating vector store
 vector store created and saved to persist directory db/chroma_db


In [ ]:
# query="did jumki loved marks?"
# retriever=db.as_retriever(search_kwargs={'k':3})
# chunks=retriever.invoke(query)


In [ ]:
# export_chunks_to_json(chunks,"rag_results.json")

In [80]:
def run_complete_ingestion_pipeline(pdf_path:str): 
    print("starting rag ingestion pipeline")

    elements=partition_document(pdf_path)
    chunks=create_chunks_by_title(elements)

    summarised_chunks=summarise_chunks(chunks)

    db=create_vector_store(summarised_chunks,persist_directory='dbv2/chroma_db')
    print("pipeline compled successfuly")
    return db
    

In [ ]:
# query="did jumki loved marks?"
# retriever=db.as_retriever(search_kwargs={'k':3})
# chunks=retriever.invoke(query)


# def generate_final_answer(chunks,query): 
#     try: 
#         llm=ChatGroq(model="llama-3.3-70b-versatile")
#         prompt_text=f"""Based on the following document, please /answet this question {query} 
#         CONTENT TO ANALYZE": """
#         for i,chunk in enumerate(chunks): 
#             prompt_text +=f'--Document {i+1} --\n'

#             if 'original_content' in chunk.metadata: 
#                 original_data=json.loads(chunks.metadata['original_content'])
#                 raw_text=original_data.get('raw_text',"")
#                 if raw_text: 
#                     prompt_text+=f'TEXT: \n {raw_text}\n\n'
                
#                 tables_html=original_data.get('tables_html',[])
#                 if tables_html: 
#                     prompt_text+="TABLES:\n"
#                     for j,table in enumerate(tables_html): 
#                         prompt_text+f"Table {j+1}:\n {table}\n\n"
#             prompt_text+="\n"
#         prompt_text+=""" 

#     please provide a clear, copmprehensive answer using the text, table and images above. If the document doesn't contain the sufficient information to answer the question, say i don't have enough information to answer the question 
     
#       ANSWER: """
#         message_content=[{"tyep":"text","text":prompt_text}]

#         for chunk in chunks: 
#             if 'original_content' in chunk.metadata: 
#                 original_data=json.load(chunk.metadata["original_content"])
#                 images_base64=original_data.get("iamges_base64",[])

#                 for image_base64 in images_base64: 
#                     message_content.append({
#                     "tyep":"image_url",
#                     "image_url":{"url":f"data:image/jpeg;base64,{image_base64}"}
#                     })
#         message=HumanMessage(content=message_content)
#         response=llm.invoke([message])
#         return response.content
#     except Exception as e: 
#         print(f"answer generation failed {e}")
#         return "sorry , i encoutered an error while generating the answer"
                


        

In [81]:
query="what is QSVD is about?" 
retriever=db.as_retriever(search_kwargs={'k':3})
chunks=retriever.invoke(query)

In [82]:
def generate_final_answer(chunks, query): 
    try: 
        llm = ChatGroq(model="llama-3.3-70b-versatile")
        prompt_text = f"""Based on the following document, please answer this question {query} 
        CONTENT TO ANALYZE": """
        
        for i, chunk in enumerate(chunks): 
            prompt_text += f'--Document {i+1} --\n'

            if 'original_content' in chunk.metadata: 
                # FIX: Access chunk instead of chunks
                original_data = json.loads(chunk.metadata['original_content'])
                raw_text = original_data.get('raw_text', "")
                if raw_text: 
                    prompt_text += f'TEXT: \n {raw_text}\n\n'
                
                tables_html = original_data.get('tables_html', [])
                if tables_html: 
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html): 
                        # FIX: Added += to correctly append
                        prompt_text += f"Table {j+1}:\n {table}\n\n"
            prompt_text += "\n"
        
        prompt_text += """ 

    please provide a clear, comprehensive answer using the text, table and images above. If the document doesn't contain the sufficient information to answer the question, say i don't have enough information to answer the question 
     
      ANSWER: """
        
        # FIX: "type" instead of "tyep"
        message_content = [{"type": "text", "text": prompt_text}]

        for chunk in chunks: 
            if 'original_content' in chunk.metadata: 
                # FIX: json.loads instead of json.load (since it's a JSON string, not a file)
                original_data = json.loads(chunk.metadata["original_content"])
                
                # FIX: "images_base64" instead of "iamges_base64"
                images_base64 = original_data.get("images_base64", [])

                for image_base64 in images_base64: 
                    message_content.append({
                        # FIX: "type" instead of "tyep"
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
                    
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        return response.content
        
    except Exception as e: 
        print(f"answer generation failed: {e}")
        return "sorry, i encountered an error while generating the answer"


In [83]:
final_answer=generate_final_answer(chunks,query)
print(final_answer)

QSVD (Quantization-Singular Value Decomposition) is a method that combines quantization and singular value decomposition (SVD) to reduce the memory usage and computational cost of large language models. It is designed to work with low-bit settings, such as W8A8 (8-bit weights and 8-bit activations) and W8A4, and can reduce the KV cache and intermediate data sizes by up to 50%.

QSVD consists of three key components:

1. Joint SVD over the combined QKV weights: This involves applying SVD to the combined QKV weights to reduce the dimensionality of the data.
2. Adaptive singular value truncation: This involves truncating the singular values to further reduce the dimensionality of the data.
3. PTQ (Post-Training Quantization) over low-rank VLMs (Vision-Language Models): This involves applying quantization to the low-rank VLMs to reduce the precision of the weights and activations.

The QSVD method has been evaluated on several benchmarks, including ScienceQA, VizWiz, and SEEDBench, and has